# Demo - Reliable ingestion: files load themselves

**Why we are doing this.** Today client files come in by SFTP, land in the data lake, and ADF and Databricks load them into Synapse. Large files queue behind each other, a client changing a column breaks the load, a bad file fails late, and reloads can create duplicates.

**What we will prove, in three passes:**
1. **Pass 1 - Parallel loads, three formats.** CSV, Avro and JSON files arrive at the same time and load at the same time, with no scheduler and no warehouse. We query all three with SQL straight away.
2. **Pass 2 - Schema evolution.** Two clients add a new column. The tables grow the column by themselves.
3. **Pass 3 - A bad file.** The good rows load, the bad rows are listed, and the ops team gets an email within a minute.

Then one quick check: reloading everything creates **no duplicates**.

**Setting.** Amazon S3 stands in for Azure Blob Storage. On Azure the SQL is identical; Event Grid replaces the S3 notification.

**Labels:** **SETUP** (already built, read only) · **RUN DURING DEMO** · **UI STEP** · **SAY THIS**

## How Snowpipe works
1. A client file lands in the storage folder.
2. Storage sends a "new file" notification to Snowflake.
3. The pipe runs its saved COPY command on just that file.
4. Rows land in the Bronze table. The file stays in storage; Snowflake only reads it.

**For the ADF and Databricks engineers:**

| What you do today | Databricks / ADF | Snowflake |
|---|---|---|
| Load files as they arrive | Auto Loader / ADF trigger | Snowpipe, `AUTO_INGEST = TRUE` |
| Handle a new column | `mergeSchema` | `ENABLE_SCHEMA_EVOLUTION = TRUE` on the table |
| Keep good rows, capture bad ones | `badRecordsPath` | `ON_ERROR = CONTINUE` + `VALIDATE_PIPE_LOAD` |
| Never load a file twice | Checkpoint location | Built-in load history (14 days per pipe) |
| Compute for loading | Cluster / integration runtime | None to manage. Serverless |

### SETUP - ALREADY DONE (read only, do not run)
**The whole pipeline for one client feed: a table and a pipe.** CLAIMS (Avro) and LABS (JSON) follow the same pattern.

```sql
CREATE TABLE BRONZE.MEMBERS_RAW (
  MEMBER_ID STRING, CLIENT_ID STRING, DOB DATE, GENDER STRING, CKD_STAGE STRING, STATE STRING,
  FILE_NAME STRING, LOADED_AT TIMESTAMP_LTZ)          -- which file each row came from, and when
ENABLE_SCHEMA_EVOLUTION = TRUE;                       -- new columns in a file are added to the table

CREATE PIPE BRONZE.PIPE_MEMBERS AUTO_INGEST = TRUE AS -- load as soon as a file arrives
COPY INTO MEMBERS_RAW FROM @S3_CLIENT_DROP/members/
  FILE_FORMAT = (FORMAT_NAME = FF_CSV)
  MATCH_BY_COLUMN_NAME = CASE_INSENSITIVE             -- match columns by name, so order does not matter
  INCLUDE_METADATA = (FILE_NAME = METADATA$FILENAME, LOADED_AT = METADATA$START_SCAN_TIME)
  ON_ERROR = CONTINUE;                                -- load the good rows, reject only the bad ones
```

> **SAY THIS:** That is the whole pipeline. No scheduler, no cluster, no trigger to maintain. Four settings do the work: auto ingest, match by name, schema evolution, and on error continue.

### RUN DURING DEMO
Set the role, database and warehouse. Run this first.

In [ ]:
%%sql -r ctx
USE ROLE ACCOUNTADMIN;
USE DATABASE SOMATUS_DEMO;
USE WAREHOUSE WH_INGEST;
USE SCHEMA BRONZE;

### RUN DURING DEMO
All three Bronze tables start empty. (If not, run 99_Reset first.)

In [ ]:
%%sql -r empty
SELECT 'MEMBERS_RAW (CSV)' AS TABLE_NAME, COUNT(*) AS ROW_COUNT FROM MEMBERS_RAW UNION ALL
SELECT 'CLAIMS_RAW (Avro)', COUNT(*) FROM CLAIMS_RAW UNION ALL
SELECT 'LABS_RAW (JSON)',   COUNT(*) FROM LABS_RAW;

## Pass 1 - Parallel loads, three formats

### UI STEP - do this in the screen, not in the notebook
**Drop three client feeds at the same time**


1. In the AWS S3 tab, open **mgupta-bucket > soybase > somatus** and click **Upload**.
2. In Finder, open **somatus-demo/drop_folders/pass1_parallel_formats**, select the three folders **members**, **claims** and **labs**, and drag them onto the upload page.
   - members: HUMANA, 4 CSV parts, 5 million rows
   - claims: 1 Avro file, 500,000 claims
   - labs: 1 nested JSON file, 200,200 lab results
3. Click **Upload**. Switch back here and wait about 45 seconds.

> **SAY THIS:** Six files, three formats, arriving together. Nobody is starting a job. Snowpipe is serverless, so these loads use none of your warehouses. A big file at ten in the morning cannot slow down Power BI. We will see what happens to the jobs that do use warehouses in the next demo.

### RUN DURING DEMO
One row per pipe. Look at the times: the three feeds started and finished together. Nothing waited for anything else. Re-run if a row count is still growing.

In [ ]:
%%sql -r parallel_loads
WITH h AS (
  SELECT 'CSV' AS FORMAT, 'MEMBERS_RAW' AS LOADED_INTO, PIPE_NAME, FILE_NAME, ROW_COUNT, PIPE_RECEIVED_TIME, LAST_LOAD_TIME
  FROM TABLE(SOMATUS_DEMO.INFORMATION_SCHEMA.COPY_HISTORY(TABLE_NAME => 'SOMATUS_DEMO.BRONZE.MEMBERS_RAW',
         START_TIME => DATEADD('hour', -1, CURRENT_TIMESTAMP())))
  UNION ALL
  SELECT 'Avro' AS FORMAT, 'CLAIMS_RAW' AS LOADED_INTO, PIPE_NAME, FILE_NAME, ROW_COUNT, PIPE_RECEIVED_TIME, LAST_LOAD_TIME
  FROM TABLE(SOMATUS_DEMO.INFORMATION_SCHEMA.COPY_HISTORY(TABLE_NAME => 'SOMATUS_DEMO.BRONZE.CLAIMS_RAW',
         START_TIME => DATEADD('hour', -1, CURRENT_TIMESTAMP())))
  UNION ALL
  SELECT 'JSON' AS FORMAT, 'LABS_RAW' AS LOADED_INTO, PIPE_NAME, FILE_NAME, ROW_COUNT, PIPE_RECEIVED_TIME, LAST_LOAD_TIME
  FROM TABLE(SOMATUS_DEMO.INFORMATION_SCHEMA.COPY_HISTORY(TABLE_NAME => 'SOMATUS_DEMO.BRONZE.LABS_RAW',
         START_TIME => DATEADD('hour', -1, CURRENT_TIMESTAMP())))
)
SELECT FORMAT, PIPE_NAME, LOADED_INTO, COUNT(*) AS FILES, SUM(ROW_COUNT) AS ROWS_LOADED,
       TO_CHAR(MIN(PIPE_RECEIVED_TIME), 'HH24:MI:SS') AS FIRST_FILE_RECEIVED,
       TO_CHAR(MAX(LAST_LOAD_TIME), 'HH24:MI:SS')     AS LAST_FILE_LOADED,
       DATEDIFF('second', MIN(PIPE_RECEIVED_TIME), MAX(LAST_LOAD_TIME)) AS SECONDS_START_TO_FINISH
FROM h GROUP BY 1, 2, 3 ORDER BY ROWS_LOADED DESC;

> **SAY THIS:** Five million members in four parts, half a million claims and two hundred thousand lab results, all loaded in parallel in under a minute. No queue.

### RUN DURING DEMO
Query each feed straight from Bronze, no transformation. **Members came from CSV.** FILE_NAME shows which client file each row came from.

In [ ]:
%%sql -r members_csv
SELECT MEMBER_ID, CLIENT_ID, DOB, CKD_STAGE, STATE, FILE_NAME
FROM MEMBERS_RAW LIMIT 5;

### RUN DURING DEMO
**Claims came from Avro.** They landed as normal typed columns.

In [ ]:
%%sql -r claims_avro
SELECT CLAIM_ID, MEMBER_ID, SERVICE_DATE, CPT_CODE, PAID_AMOUNT, FILE_NAME
FROM CLAIMS_RAW LIMIT 5;

### RUN DURING DEMO
**Labs came from nested JSON.** Each record is stored whole in one column, RAW, and read with dot notation. (The filter skips a few deliberately bad records we use in the data quality demo.)

In [ ]:
%%sql -r labs_json
SELECT RAW:member_id::STRING             AS MEMBER_ID,
       RAW:panel.results[0].test::STRING  AS TEST,
       RAW:panel.results[0].value::FLOAT  AS VALUE,
       FILE_NAME
FROM LABS_RAW
WHERE RAW:member_id IS NOT NULL AND RAW:panel.results[0].value::FLOAT > 0
LIMIT 5;

> **SAY THIS:** Here it is. Members in CSV, claims in Avro and labs in JSON, all queryable right away without writing any transformation. And we can even join them together to analyse the data.

### RUN DURING DEMO
Now join all three formats in one query, still straight from Bronze.

In [ ]:
%%sql -r three_formats
WITH egfr AS (                                         -- nested JSON: one row per eGFR result
  SELECT RAW:member_id::STRING AS MEMBER_ID
  FROM LABS_RAW, LATERAL FLATTEN(RAW:panel.results) r
  WHERE r.value:test::STRING = 'eGFR'
)
SELECT m.CKD_STAGE,
       COUNT(DISTINCT m.MEMBER_ID) AS MEMBERS_FROM_CSV,
       COUNT(DISTINCT c.CLAIM_ID)  AS CLAIMS_FROM_AVRO,
       COUNT(DISTINCT e.MEMBER_ID) AS MEMBERS_WITH_EGFR_FROM_JSON
FROM MEMBERS_RAW m
LEFT JOIN CLAIMS_RAW c ON c.MEMBER_ID = m.MEMBER_ID
LEFT JOIN egfr e       ON e.MEMBER_ID = m.MEMBER_ID
GROUP BY 1 ORDER BY 1;

> **SAY THIS:** Today JSON and CSV go to separate folders and separate processes. Here the JSON is stored as it arrived and queried right next to the CSV and Avro data, with one line of SQL. You decide later which fields to promote into Silver.

## Pass 2 - Schema evolution: clients add columns

### RUN DURING DEMO
**Columns before.** The columns of MEMBERS_RAW and CLAIMS_RAW today. Note there is no DIALYSIS_MODALITY in MEMBERS_RAW and no PLACE_OF_SERVICE in CLAIMS_RAW.

In [ ]:
%%sql -r schema_before
SELECT TABLE_NAME, COLUMN_NAME, DATA_TYPE
FROM SOMATUS_DEMO.INFORMATION_SCHEMA.COLUMNS
WHERE TABLE_SCHEMA = 'BRONZE' AND TABLE_NAME IN ('MEMBERS_RAW', 'CLAIMS_RAW')
ORDER BY TABLE_NAME DESC, ORDINAL_POSITION;

> **SAY THIS:** Remember these two definitions. Two clients are about to send files with a column that is not in them.

### UI STEP - do this in the screen, not in the notebook
**Drop the changed files**


1. In the S3 tab, go back to **somatus** and click **Upload**.
2. From **drop_folders/pass2_schema_change**, drag the folders **members** and **claims** onto the upload page.
   - members: AETNA, columns in a different order **plus a new column DIALYSIS_MODALITY**
   - claims: version 2 Avro with **a new field PLACE_OF_SERVICE**
3. Click **Upload** and wait about 30 seconds.

### RUN DURING DEMO
**Columns after.** The same query again. MEMBERS_RAW now ends with **DIALYSIS_MODALITY** and CLAIMS_RAW with **PLACE_OF_SERVICE**. Nobody ran ALTER TABLE.

In [ ]:
%%sql -r schema_after
SELECT TABLE_NAME, COLUMN_NAME, DATA_TYPE
FROM SOMATUS_DEMO.INFORMATION_SCHEMA.COLUMNS
WHERE TABLE_SCHEMA = 'BRONZE' AND TABLE_NAME IN ('MEMBERS_RAW', 'CLAIMS_RAW')
ORDER BY TABLE_NAME DESC, ORDINAL_POSITION;

> **SAY THIS:** Same tables, one new column each. The files brought the columns, and the tables added them, because schema evolution is switched on. Now let's check the data in them.

### RUN DURING DEMO
Both new columns now exist and hold data. Nobody ran ALTER TABLE. Older rows show the new column as empty, which is correct.

In [ ]:
%%sql -r new_columns
SELECT 'MEMBERS_RAW (CSV)' AS TABLE_NAME, 'DIALYSIS_MODALITY' AS NEW_COLUMN,
       COUNT(*) AS TOTAL_ROWS, COUNT(DIALYSIS_MODALITY) AS ROWS_WITH_VALUE, COUNT(*) - COUNT(DIALYSIS_MODALITY) AS ROWS_EMPTY
FROM MEMBERS_RAW
UNION ALL
SELECT 'CLAIMS_RAW (Avro)', 'PLACE_OF_SERVICE',
       COUNT(*), COUNT(PLACE_OF_SERVICE), COUNT(*) - COUNT(PLACE_OF_SERVICE)
FROM CLAIMS_RAW;

### RUN DURING DEMO
Who brought the new column, and the values it carries.

In [ ]:
%%sql -r new_col_values
SELECT CLIENT_ID, DIALYSIS_MODALITY, COUNT(*) AS MEMBERS
FROM MEMBERS_RAW GROUP BY 1, 2 ORDER BY 1, 3 DESC;

> **SAY THIS:** Two clients changed their layout without telling us. In a position-based pipeline that breaks the load or silently shifts every column. Here the columns matched by name, the tables grew the new columns, and nobody was paged.

## Pass 3 - A bad file: good rows load, bad rows are reported, ops gets an email

### UI STEP - do this in the screen, not in the notebook
**Drop the CIGNA file with bad dates**


1. In the S3 tab, go back to **somatus** and click **Upload**.
2. From **drop_folders/pass3_bad_file**, drag the folder **members** onto the upload page. The CIGNA file has 100 rows; 10 have the text 'not-a-date' as the date of birth.
3. Click **Upload** and wait about 30 seconds.

### RUN DURING DEMO
The load result for the CIGNA file: partially loaded, 100 in the file, 90 loaded, 10 rejected.

In [ ]:
%%sql -r cigna_load
SELECT FILE_NAME, STATUS, ROW_PARSED AS ROWS_IN_FILE, ROW_COUNT AS ROWS_LOADED,
       ERROR_COUNT AS ROWS_REJECTED, FIRST_ERROR_MESSAGE
FROM TABLE(INFORMATION_SCHEMA.COPY_HISTORY(TABLE_NAME => 'MEMBERS_RAW',
           START_TIME => DATEADD('hour', -1, CURRENT_TIMESTAMP())))
WHERE FILE_NAME ILIKE '%CIGNA%';

### RUN DURING DEMO
Every rejected row: line number, column, reason and the original record.

In [ ]:
%%sql -r rejected_rows
SELECT LINE, REGEXP_SUBSTR(COLUMN_NAME, '\\["([A-Z_0-9]+)"', 1, 1, 'e') AS COLUMN_NAME, ERROR, REJECTED_RECORD
FROM TABLE(INFORMATION_SCHEMA.VALIDATE_PIPE_LOAD(
       PIPE_NAME  => 'SOMATUS_DEMO.BRONZE.PIPE_MEMBERS',
       START_TIME => DATEADD('hour', -1, CURRENT_TIMESTAMP())))
ORDER BY LINE;

> **SAY THIS:** One bad value did not stop the load. The 90 good rows are already usable. ON_ERROR is a per-pipe choice: continue like this, skip the whole file, or skip it only past an error threshold.

### SETUP - ALREADY DONE (read only, do not run)
**The load error alert.** A serverless alert checks the load history every minute. When a file has rejected rows that nobody has been told about, it calls a procedure that emails the ops team.

```sql
CREATE ALERT OPS.ALERT_LOAD_ERRORS
  SCHEDULE = '1 MINUTE'                               -- serverless: no warehouse
  IF (EXISTS (SELECT 1 FROM TABLE(INFORMATION_SCHEMA.COPY_HISTORY(TABLE_NAME => 'BRONZE.MEMBERS_RAW', ...))
              WHERE ERROR_COUNT > 0 AND <file not reported yet>))
  THEN CALL OPS.SP_NOTIFY_LOAD_ERRORS();

-- SP_NOTIFY_LOAD_ERRORS, for each new bad file:
--   1. reads the rejected rows (VALIDATE_PIPE_LOAD)
--   2. writes them all to a CSV on a stage and creates a 7-day download link (GET_PRESIGNED_URL)
--   3. emails a summary, the reasons, the first 20 rows and the link (SYSTEM$SEND_EMAIL, HTML)
--   4. logs the file so it is reported only once
```

### RUN DURING DEMO
Fire the alert now instead of waiting for its next one-minute check. Then switch to the inbox.

In [ ]:
%%sql -r fire_alert
EXECUTE ALERT SOMATUS_DEMO.OPS.ALERT_LOAD_ERRORS;

### UI STEP - do this in the screen, not in the notebook
**Show the email**


1. Open your inbox. The email is from **no-reply@snowflake.net**, subject **Load issue: members_CIGNA_DEMO_20261022_bad.csv - 10 of 100 rows rejected**.
2. Point at: the rows in file / loaded / rejected, the reason summary, and the rejected rows table.
3. Click **Download all rejected rows (CSV)** to show the full list.

### RUN DURING DEMO
If the email is slow: the alert history shows it fired, and the log shows the file was reported.

In [ ]:
%%sql -r alert_check
SELECT NAME, STATE, SCHEDULED_TIME, COMPLETED_TIME
FROM TABLE(INFORMATION_SCHEMA.ALERT_HISTORY(SCHEDULED_TIME_RANGE_START => DATEADD('minute', -15, CURRENT_TIMESTAMP())))
WHERE NAME = 'ALERT_LOAD_ERRORS' ORDER BY SCHEDULED_TIME DESC LIMIT 3;

> **SAY THIS:** Today a failure can surface two hours later or the next morning. Here the ops team gets this within a minute of the file landing: what happened, why, and the exact rows to send back to the client.

## Finally - Reload everything: no duplicates

### RUN DURING DEMO
Row counts before the reload. You should see:

| Table | Rows | Made up of |
|---|---|---|
| MEMBERS_RAW | **5,050,090** | HUMANA 5,000,000 + AETNA 50,000 + CIGNA 90 good rows |
| CLAIMS_RAW | **600,000** | v1 500,000 + v2 100,000 |
| LABS_RAW | **200,200** | one JSON file |

In [ ]:
%%sql -r counts
SELECT 'MEMBERS_RAW' AS TABLE_NAME, COUNT(*) AS ROW_COUNT FROM MEMBERS_RAW UNION ALL
SELECT 'CLAIMS_RAW', COUNT(*) FROM CLAIMS_RAW UNION ALL
SELECT 'LABS_RAW',   COUNT(*) FROM LABS_RAW;

### RUN DURING DEMO
Ask every pipe to re-scan its whole folder, as an engineer might after an outage. Wait about 30 seconds, then run the next cell.

In [ ]:
%%sql -r reload_all
ALTER PIPE PIPE_MEMBERS REFRESH;
ALTER PIPE PIPE_CLAIMS  REFRESH;
ALTER PIPE PIPE_LABS    REFRESH;

### RUN DURING DEMO
Row counts after the reload. They should be exactly the same: **5,050,090** members, **600,000** claims, **200,200** labs.

In [ ]:
%%sql -r counts_after
SELECT 'MEMBERS_RAW' AS TABLE_NAME, COUNT(*) AS ROW_COUNT FROM MEMBERS_RAW UNION ALL
SELECT 'CLAIMS_RAW', COUNT(*) FROM CLAIMS_RAW UNION ALL
SELECT 'LABS_RAW',   COUNT(*) FROM LABS_RAW;

> **SAY THIS:** Same counts. Each pipe remembers which files it already loaded and skips them. Reprocessing is safe by default. A forced reload is possible, but only as a deliberate choice.

## What we just proved

| # | Your challenge today | What you saw |
|---|---|---|
| 1 | Large files queue behind each other | Six files across three feeds loaded **in parallel**, start to finish in under a minute: **5,000,000 members** (CSV, 4 parts), **500,000 claims** (Avro) and **200,200 lab results** (JSON) |
| 2 | Loads compete with reporting for compute | Snowpipe is **serverless**. Loading used none of your warehouses, so it cannot slow Power BI |
| 3 | Pipelines to build and schedule | **One table and one pipe per feed**. No scheduler, no cluster, no trigger |
| 4 | CSV, Avro and JSON handled separately | All three formats were **queried straight away with SQL**, on their own and joined together, with no transformation first |
| 5 | A client adds or reorders columns and the load breaks | **Schema evolution** added DIALYSIS_MODALITY (AETNA, 50,000 members) and PLACE_OF_SERVICE (claims v2, 100,000 claims). Columns matched by name |
| 6 | One bad value fails the whole file | CIGNA: **90 good rows loaded**, **10 bad rows** listed with line, column and reason |
| 7 | Failures found hours later | **Email within a minute**, with a reason summary, the bad rows and a CSV download link |
| 8 | Reloads create duplicates | Re-scanning every folder loaded nothing twice: **5,050,090 / 600,000 / 200,200** before and after |

**Next:** what happens to the jobs that *do* use warehouses: loads, reporting and Power BI running at the same time.

### IF ASKED
- **"Can we fail the whole file instead?"** Yes. `ON_ERROR = SKIP_FILE`, or skip only past a count or percentage of errors.
- **"Corrected file with the same name?"** Pipes track files by path, so resend with a new name (a version or timestamp suffix). Silver keeps the latest record per member.
- **"How long does the pipe remember files?"** 14 days of load history per pipe.
- **"Can the email attach the file?"** Snowflake emails do not carry attachments, so the full list is a 7-day download link. On Azure you can also push pipe errors to Event Grid and into your existing alerting.
- **"Cost?"** Snowpipe is billed per file loaded and the serverless compute used, not for idle time.